In [1]:
import json
import boto3
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
endpoint = os.getenv("S3_ENDPOINT", os.getenv("SEAWEEDFS_S3_ENDPOINT", "http://127.0.0.1:8333"))
if endpoint and not str(endpoint).startswith("http"):
    endpoint = f"http://{endpoint}"

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID", "admin"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY", "admin"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
)
print("S3 endpoint:", endpoint)

S3 endpoint: http://seaweedfs:8333


**JSON**

In [3]:
# Coger el JSON de esta "carpeta" y imprimirlo (si hay varios, usa el primero; cambia a keys[-1] si quieres el último)
bucket = "landing-zone"
prefix = "transit_posts"  # misma ruta que listas arriba; acaba en /

keys = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        k = obj["Key"]
        if k.endswith(".json"):
            keys.append(k)

if not keys:
    raise FileNotFoundError(f"No hay ningún .json bajo s3://{bucket}/{prefix}")

key = keys[0]
print("Leyendo:", key, "\n")

raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
data = json.loads(raw.decode("utf-8"))
print(json.dumps(data, indent=2, ensure_ascii=False))

Leyendo: transit_posts/post_2026-05-29T13:45:24+00:00.json 

{
  "title": "🔁 Pas alternatiu a l'N-340 a l'Arboç a causa d'un #accident \n\n🔴 #Retencions en ambdós sentits, sobretot en sentit Tarragona",
  "title_detail": {
    "type": "text/plain",
    "language": null,
    "base": "https://nitter.net/transit/rss",
    "value": "🔁 Pas alternatiu a l'N-340 a l'Arboç a causa d'un #accident \n\n🔴 #Retencions en ambdós sentits, sobretot en sentit Tarragona"
  },
  "authors": [
    {
      "name": "@transit"
    }
  ],
  "author": "@transit",
  "author_detail": {
    "name": "@transit"
  },
  "summary": "<p>🔁 Pas alternatiu a l'N-340 a l'Arboç a causa d'un <a href=\"https://nitter.net/search?f=tweets&amp;q=%23accident\">#accident</a> <br />\n<br />\n🔴 <a href=\"https://nitter.net/search?f=tweets&amp;q=%23Retencions\">#Retencions</a> en ambdós sentits, sobretot en sentit Tarragona</p>\n<img src=\"https://nitter.net/pic/media%2FHJfbmVoWsAIJtA9.jpg\" />",
  "summary_detail": {
    "type": "tex

**CSV**

In [5]:
import pandas as pd
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "trusted-zone"
prefix = "ajuntament_bcn/trams_congestio"

# Llista tots els CSVs
csvs = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".csv"):
            csvs.append(obj["Key"])

print(f"CSVs trobats: {len(csvs)}")

# Llegeix tots i ajunta'ls en un sol DataFrame
dfs = []
for key in csvs:
    raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
    df = pd.read_csv(io.BytesIO(raw))
    print(f"  → {key}: {len(df)} files -> columns: {df.columns} -> numero de nulos: {df.isna().sum()}")
    dfs.append(df)

# Ajunta tots
df_total = pd.concat(dfs, ignore_index=True)
print(f"\nTotal files combinades: {len(df_total)}")
print(df_total)


CSVs trobats: 0


ValueError: No objects to concatenate

**XML**

In [30]:
import boto3

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "landing-zone"
prefix = "sct/traffic_incidents/"

# Llista tots els XMLs
xmls = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".xml"):
            xmls.append(obj["Key"])

print(f"XMLs trobats: {len(xmls)}")

# Llegeix i imprimeix cada un
for key in xmls:
    print(f"\n{'='*50}")
    print(f"Fitxer: {key}")
    print('='*50)
    raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
    print(raw.decode("utf-8"))

XMLs trobats: 1

Fitxer: sct/traffic_incidents/2026-05-06T23:25:00+00:00.xml
<?xml version="1.0" encoding="UTF-8"?><wfs:FeatureCollection xmlns="http://www.opengis.net/wfs" xmlns:wfs="http://www.opengis.net/wfs" xmlns:gml="http://www.opengis.net/gml" xmlns:cite="http://www.opengeospatial.net/cite" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengeospatial.net/cite http://localhost:8080/sct-gis/wfs?service=WFS&amp;version=1.0.0&amp;request=DescribeFeatureType&amp;typeName=cite%3Amct2_v_afectacions_data http://www.opengis.net/wfs http://localhost:8080/sct-gis/schemas/wfs/1.0.0/WFS-basic.xsd"><gml:boundedBy><gml:null>unknown</gml:null></gml:boundedBy><gml:featureMember><cite:mct2_v_afectacions_data fid="mct2_v_afectacions_data.fid-1f259c63_19dfd8abf28_6ac9"><cite:geom><gml:Point srsName="http://www.opengis.net/gml/srs/epsg.xml#4326"><gml:coordinates xmlns:gml="http://www.opengis.net/gml" decimal="." cs="," ts=" ">1.98161463,41.4605575</gml:coordina

**PERSISTENT-LANDING-ZONE**

In [31]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "persistent-landing-zone"
prefix = "sct/traffic_incidents/"


parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.dtypes)
    

Parquets trobats: 1
Total files: 1
Index(['file_name', 'xml_content'], dtype='str')
                                           file_name  \
0  sct/traffic_incidents/2026-05-06T23:25:00+00:0...   

                                         xml_content  
0  <?xml version="1.0" encoding="UTF-8"?><wfs:Fea...  
file_name      str
xml_content    str
dtype: object


**TRUSTED ZONE**

In [11]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "trusted-zone"
prefix = "ajuntament_bcn/accidents_gu/2021_ACCIDENTS_GU_BCN/"


parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.dtypes)

    

Parquets trobats: 7
Total files: 1078
Index(['Numero_expedient', 'Codi_districte', 'Nom_districte', 'Codi_barri',
       'Nom_barri', 'Codi_carrer', 'Nom_carrer', 'Num_postal_caption',
       'Descripcio_dia_setmana', 'NK_Any', 'Mes_any', 'Nom_mes', 'Dia_mes',
       'Hora_dia', 'Descripcio_torn', 'Descripcio_causa_vianant',
       'Numero_morts', 'Numero_lesionats_lleus', 'Numero_lesionats_greus',
       'Numero_victimes', 'Numero_vehicles_implicats', 'Coordenada_UTM_X_ED50',
       'Coordenada_UTM_Y_ED50', 'Longitud_WGS84', 'Latitud_WGS84'],
      dtype='str')
  Numero_expedient  Codi_districte         Nom_districte  Codi_barri  \
0      2021S000008               5  SarriÃ -Sant Gervasi          26   
1      2021S000012               2              Eixample           7   
2      2021S000018               6               GrÃ cia          32   
3      2021S000022               2              Eixample           8   
4      2021S000032               2              Eixample           7   